In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import preprocessing
import matplotlib.pyplot as plt
%matplotlib inline

import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_predict

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import cross_val_score

In [16]:
dfa=pd.read_csv("prepared_obs.csv")
dfb=pd.read_csv('prepared_plant2_obs.csv')
df_pest= pd.read_csv('prepared_pest_obs.csv')

In [4]:
dfa.head(30)

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome
0,315987576,2025/09/22 1:15 PM,49.230847,-122.929039,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
1,315797997,2025-09-22 14:11:58,55.324646,-1.554149,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
2,316026631,2025-09-22 14:10:16+02:00,60.718503,-46.036708,Flowering,Achillea millefolium,common yarrow,52821,265,29.0
3,315784690,2025-09-22 13:51:14,48.200201,17.196155,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
4,315774789,2025-09-22 11:16:34,50.773417,-3.999461,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
5,315797416,2025/09/22 10:24 AM,53.015837,-1.217560,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
6,315889887,2025/09/22 9:49 AM,52.633048,-2.159995,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
7,315796359,2025-09-21 13:47:52,38.693507,-111.680819,Flowering,Achillea millefolium,common yarrow,52821,264,26.0
8,315692619,2025-09-21 14:36:17,42.952977,-81.220301,Flowering,Achillea millefolium,common yarrow,52821,264,26.0
9,315575227,2025-09-21 14:11:03,61.149895,-45.514545,Flowering,Achillea millefolium,common yarrow,52821,264,29.0


In [5]:
dfb.head(30)

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome
0,299753119,2025-07-21 16:51:12,-39.434370,176.601988,Fruiting,Melicytus ramiflorus,Māhoe,197063,202,15.0
1,284940738,2025-05-29 08:38:15,-36.863667,174.803194,Flowering,Melicytus ramiflorus,Māhoe,197063,149,15.0
2,281118698,2025-05-11 14:06:12+12:00,-43.623842,172.638870,Fruiting,Melicytus ramiflorus,Māhoe,197063,131,15.0
3,280156036,2025-05-11 13:45:12,-40.191279,176.135594,Fruiting,Melicytus ramiflorus,Māhoe,197063,131,15.0
4,273582061,2025-04-26 11:16:37,-36.810763,174.943681,Flowering,Melicytus ramiflorus,Māhoe,197063,116,15.0
5,277585585,2025/04/26 10:05 AM,-43.579545,172.633167,Fruiting,Melicytus ramiflorus,Māhoe,197063,116,15.0
6,272136460,2025-04-22 17:13:17,-43.619107,172.650201,Fruiting,Melicytus ramiflorus,Māhoe,197063,112,15.0
7,272034640,2025-04-21 21:10:51,-38.517022,175.582863,Fruiting,Melicytus ramiflorus,Māhoe,197063,111,15.0
8,279300214,2025-04-18 12:45:40,-41.298378,174.010819,Fruiting,Melicytus ramiflorus,Māhoe,197063,108,15.0
9,277671049,2025-04-06 14:15:39+12:00,-45.842037,170.660478,Fruiting,Melicytus ramiflorus,Māhoe,197063,96,15.0


In [29]:
# mergeing the two data sets 
plant = pd.concat([dfa,dfb], axis=0, ignore_index=True)
plant.head()

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome
0,315987576,2025/09/22 1:15 PM,49.230847,-122.929039,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
1,315797997,2025-09-22 14:11:58,55.324646,-1.554149,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
2,316026631,2025-09-22 14:10:16+02:00,60.718503,-46.036708,Flowering,Achillea millefolium,common yarrow,52821,265,29.0
3,315784690,2025-09-22 13:51:14,48.200201,17.196155,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
4,315774789,2025-09-22 11:16:34,50.773417,-3.999461,Flowering,Achillea millefolium,common yarrow,52821,265,15.0


In [32]:
plant.shape

(758759, 10)

In [31]:
import pandas as pd
import re

# Example: ensure column is string
plant['observed_on'] = plant['observed_on'].astype(str)

# Extract only the date portion using regex
plant['observed_on'] = plant['observed_on'].apply(
    lambda x: re.search(r'\d{4}[-/]\d{2}[-/]\d{2}', x).group(0) if re.search(r'\d{4}[-/]\d{2}[-/]\d{2}', x) else x
)

# Standardize format (replace '/' with '-')
plant['observed_on'] = plant['observed_on'].str.replace('/', '-', regex=False)


In [ ]:
# Drop the temporary parsed column
plant.drop(columns='observed_on_parsed', inplace=True)


In [34]:
plant.head(30)

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome
0,315987576,2025-09-22,49.230847,-122.929039,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
1,315797997,2025-09-22,55.324646,-1.554149,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
2,316026631,2025-09-22,60.718503,-46.036708,Flowering,Achillea millefolium,common yarrow,52821,265,29.0
3,315784690,2025-09-22,48.200201,17.196155,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
4,315774789,2025-09-22,50.773417,-3.999461,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
5,315797416,2025-09-22,53.015837,-1.217560,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
6,315889887,2025-09-22,52.633048,-2.159995,Flowering,Achillea millefolium,common yarrow,52821,265,15.0
7,315796359,2025-09-21,38.693507,-111.680819,Flowering,Achillea millefolium,common yarrow,52821,264,26.0
8,315692619,2025-09-21,42.952977,-81.220301,Flowering,Achillea millefolium,common yarrow,52821,264,26.0
9,315575227,2025-09-21,61.149895,-45.514545,Flowering,Achillea millefolium,common yarrow,52821,264,29.0


In [39]:
import numpy as np

conditions = [
    (plant['biome'].between(1, 5)),
    (plant['biome'].between(6, 8)),
    (plant['biome'].between(9, 14)),
    (plant['biome'].between(15, 21)),
    (plant['biome'].between(22, 24)),
    (plant['biome'] >= 25)
]

choices = ['A', 'B', 'C', 'D', 'E', 'H']

plant['biome_cat'] = np.select(conditions, choices, default='Unknown')


In [44]:
plant.head()

,observed_on,phenophase,scientific_name,common_name,doy,biome,biome_cat
0,2025-09-22,Flowering,Achillea millefolium,common yarrow,265,15.0,D
1,2025-09-22,Flowering,Achillea millefolium,common yarrow,265,15.0,D
2,2025-09-22,Flowering,Achillea millefolium,common yarrow,265,29.0,H
3,2025-09-22,Flowering,Achillea millefolium,common yarrow,265,15.0,D
4,2025-09-22,Flowering,Achillea millefolium,common yarrow,265,15.0,D


In [43]:
plant.drop(columns=['id','latitude','longitude','taxon_id'],inplace=True)

In [48]:
plant['doy'].head(20)

0     265
1     265
2     265
3     265
4     265
5     265
6     265
7     264
8     264
9     264
10    264
11    264
12    264
13    264
14    264
15    264
16    264
17    262
18    261
19    260
Name: doy, dtype: int64